# 01 | Source inventory and ingestion
**NHS A&E Intelligent Decision Support System**

**Objective:** verify monthly source coverage and construct a traceable staging dataset for provider-level Type 1 forecasting.

**Source:** [NHS England A&E activity](https://www.england.nhs.uk/statistics/statistical-work-areas/ae-waiting-times-and-activity/) and the repository's original CSVs (January 2022 to September 2026). National totals and all original observations remain in staging.

**Execution:** explore reads saved evidence; rebuild runs the stages in an isolated local workspace; snapshot generates a versioned research run. See the project README for the mode settings.

In [ ]:
from pathlib import Path
import sys

MODE = "explore"  # explore, rebuild, snapshot
VIEW_REBUILD = False  # In explore mode, view completed local rebuild results.
RESET_REBUILD = False  # Notebook 01 only: explicitly replace the local rebuild.
EXPLORE_RUN_IDS = {}  # Optional saved run IDs by milestone, e.g. {"M03": "..."}.

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "src/utils/execution_modes.py").is_file()), None)
if project_root is None:
    raise FileNotFoundError("Open the project repository before running this notebook.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from src.utils.execution_modes import NotebookSession

session = NotebookSession(project_root, MODE, 1, reset_rebuild=RESET_REBUILD,
                          view_rebuild=VIEW_REBUILD, run_ids=EXPLORE_RUN_IDS)
repo_root = session.repo
print("Execution mode:", MODE)
if MODE == "explore":
    print("Saved evidence:", session.locations())
else:
    from pathlib import Path
    import sys, json
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display, Markdown

    from src.utils.evidence_exporter import EvidenceRun, record_common_inputs, M01_REPORTS, M02_REPORTS
    RUN_TESTS = True

    from src.data.data_ingestion import run_pipeline
    START_MONTH = '2022-01'
    END_MONTH = '2026-09'


## 2. Build the staging dataset
Read the 57 monthly CSVs, verify the source inventory and preserve the original 22 fields with four provenance fields. Only empty trailing columns may be excluded; unresolved schema or coverage errors stop ingestion.

In [ ]:
if MODE == "explore":
    session.show('M01', 'reports/m01_coverage.json')
else:
    ingestion = run_pipeline(repo_root, start=START_MONTH, end=END_MONTH)
    display(pd.DataFrame([ingestion['m01']])[['source_files','unique_content_months','preliminary_review_files','total_csv_data_rows']])


## 3. Reporting coverage
Count files by reporting year and inspect coverage and source-format exceptions. This figure measures file availability, not patient demand.

In [ ]:
if MODE == "explore":
    session.show('M01', 'figures/01_source_coverage.png')
    session.show('M01', 'reports/inventory_interpretation.md')
else:
    m01 = EvidenceRun(repo_root, "M01", "Archive completed source inventory and visualise saved coverage.")
    record_common_inputs(m01)
    m01.record_input("notebooks/01_data_ingestion.ipynb")
    for path in M01_REPORTS:
        m01.copy_report(path)
    manifest = pd.read_csv(m01.path / "reports/source_manifest.csv", dtype=str, keep_default_na=False)
    coverage = json.loads((m01.path / "reports/m01_coverage.json").read_text(encoding="utf-8"))
    assert len(manifest) == coverage["source_files"], "Source inventory and coverage counts disagree."
    counts = manifest.loc[manifest["content_month"].ne(""), "content_month"].str[:4].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(9, 4.2), layout="constrained")
    bars = ax.bar(counts.index, counts.values, color="#245A81")
    ax.bar_label(bars, padding=4)
    ax.set(title=f"Source inventory: {len(manifest)} monthly CSV files",
           xlabel="Reporting year", ylabel="Files with identified reporting month", ylim=(0, max(12, counts.max()) + 2))
    ax.grid(axis="y", alpha=.2)
    m01.save_figure(fig, "01_source_coverage.png")
    plt.show()
    review_m01 = bool(coverage["missing_months"] or coverage["duplicate_months"] or coverage["unexpected_months"] or coverage["preliminary_review_files"])
    text = (f"**Observed:** {coverage['source_files']} files; {coverage['unique_content_months']} distinct reporting months; "
            f"{len(coverage['missing_months'])} missing months; {coverage['preliminary_review_files']} files requiring review; "
            f"{coverage.get('source_format_warning_files', 0)} source-format warnings.\n\n"
            "**Decision:** " + ("Investigate before accepting ingestion." if review_m01 else "Retain the original source files and review documented format warnings; use the staged data for detailed validation."))
    display(Markdown(text))
    m01.write_text("reports/inventory_interpretation.md", text)


## 4. Retained source observations
Compare the provider and national record groups. Retain both in staging until standardisation.

In [ ]:
if MODE == "explore":
    session.show('M02', 'figures/01_ingested_record_groups.png')
    session.show('M02', 'reports/ingestion_interpretation.md')
else:
    m02 = EvidenceRun(repo_root, "M02", "Archive completed ingestion summary and visualise retained record groups.")
    record_common_inputs(m02, include_staged=True)
    m02.record_input("notebooks/01_data_ingestion.ipynb")
    for path in M02_REPORTS:
        m02.copy_report(path)
    summary = json.loads((m02.path / "reports/m02_ingestion_summary.json").read_text(encoding="utf-8"))
    provider, aggregate = summary["provider_rows"], summary["national_total_rows_retained"]
    review_m02 = (summary["combined_rows_including_totals"] != coverage["total_csv_data_rows"] or
                  provider != coverage["total_provider_rows"] or aggregate != coverage["total_aggregate_rows"] or
                  provider + aggregate != summary["combined_rows_including_totals"])
    counts = pd.Series({"Provider records": provider, "National TOTAL records": aggregate})
    fig, ax = plt.subplots(figsize=(9, 3.8), layout="constrained")
    bars = ax.barh(counts.index, counts.values, color=["#245A81", "#B56B13"])
    ax.invert_yaxis()
    ax.bar_label(bars, labels=[f"{x:,}" for x in counts], padding=5)
    ax.set(title=f"Ingestion retained {summary['combined_rows_including_totals']:,} records",
           xlabel="Recorded rows (count)", xlim=(0, max(1, counts.max()) * 1.18))
    ax.grid(axis="x", alpha=.2)
    m02.save_figure(fig, "01_ingested_record_groups.png")
    plt.show()
    text = (f"**Observed:** {provider:,} provider rows and {aggregate:,} national totals. "
            f"{summary['original_source_columns']} source fields plus {summary['additive_provenance_columns']} provenance fields.\n\n"
            "**Count comparison:** " + ("Review required." if review_m02 else "Saved inventory and ingestion counts agree.") +
            "\n\n**Decision:** Retain the staged data unchanged; validate all numerical measures and provider-month keys at M03.")
    display(Markdown(text))
    m02.write_text("reports/ingestion_interpretation.md", text)


## 5. Verification and evidence
Run regression tests and save the figures, reports and input fingerprints. Proceed to M03 only after successful inventory and ingestion.

In [ ]:
if MODE == "explore":
    print('Saved evidence:', session.locations())
else:
    if RUN_TESTS:
        for run in (m01, m02):
            rc = run.run_tests("tests/test_data_ingestion.py")
            print(run.milestone, "pytest exit code:", rc)
    for run, review in ((m01, review_m01), (m02, review_m02)):
        path = run.finish("Existing reports preserved byte-for-byte. Charts use saved report values. Tests, if enabled, are fresh synthetic-fixture tests.", review)
        print(run.milestone, path.relative_to(repo_root))
        print(json.loads((path / "evidence_manifest.json").read_text(encoding="utf-8"))["status"])
    session.finish()
    print('Stage completed.')
